# GuardSLM — Open-Source SLM Safety Guard Evaluation (Google Colab)

This notebook provides an out-of-the-box execution pipeline for evaluating free, open-source Safety Language Models (SLMs) on **40 matched pairs / 80 controlled test cases**.

Supported Open-Source Models:
- **Qwen Guard** (`Qwen/Qwen2.5-1.5B-Instruct` / `Qwen/Qwen2.5-7B-Instruct`)
- **Llama Guard 3** (`meta-llama/Llama-Guard-3-1B` / `meta-llama/Llama-Guard-3-8B`)
- **WebGuard** (`OSU-NLP/WebGuard-7B`)
- **DynaGuard** (`DynaGuard/DynaGuard-8B`)
- **PolicyGuard** (`PolicyGuard/PolicyGuard-4B`)
- **Rule Baseline** (Deterministic state rules)

### Step 1 — Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

# If running in Google Colab, clone repo
if os.path.exists('/content'):
    %cd /content
    if not os.path.exists('GuardSLM'):
        !git clone https://github.com/AkarshiAaryan/GuardSLM.git
    %cd GuardSLM

sys.path.insert(0, os.getcwd())
print(f"Project root: {os.getcwd()}")

### Step 2 — Install Free Open-Source Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers torch accelerate bitsandbytes

### Step 3 — Hardware & VRAM Verification

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU. Enable GPU acceleration in Colab: Runtime -> Change runtime type -> T4 GPU.")

### Step 4 — Validate Dataset & Schema

In [ ]:
from src.data.validator import validate_dataset_file
from src.data.loader import load_test_cases

dataset_path = 'data/dummy/dummy_cases.json'
is_valid, errors = validate_dataset_file(dataset_path)

if is_valid:
    cases = load_test_cases(dataset_path)
    print(f"SUCCESS: Validated {len(cases)} test cases across {len(set(c.pair_id for c in cases))} matched pairs!")
else:
    print("Validation errors:", errors)

### Step 5 — Run Open-Source SLM Safety Guard (e.g. Qwen2.5-1.5B / Qwen2.5-7B)

In [ ]:
from src.models.qwen_guard import QwenGuardAdapter
from src.baselines.rule_baseline import RuleBaseline
from src.evaluation.runner import run_evaluation_for_model

# Pre-configured open-source checkpoint
qwen_config = {
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-1.5B-Instruct",  # Or "Qwen/Qwen2.5-7B-Instruct"
    "load_in_4bit": False
}

qwen_guard = QwenGuardAdapter(name="qwen_guard", config=qwen_config)
results = run_evaluation_for_model(qwen_guard, cases)
print(f"Completed {len(results)} predictions with Qwen Guard!")

### Step 6 — Calculate Accuracy, Pair Accuracy & Context Flip Rate

In [ ]:
from src.evaluation.metrics import calculate_overall_metrics
from src.evaluation.pair_metrics import calculate_pair_metrics
from src.evaluation.failure_analysis import analyze_failures

overall = calculate_overall_metrics(results)
pair_m = calculate_pair_metrics(results)
failures = analyze_failures(results)

print(f"Overall Accuracy:  {overall['overall_accuracy']*100:.1f}%")
print(f"Pair Accuracy:     {pair_m['pair_accuracy']*100:.1f}%")
print(f"Context Flip Rate: {pair_m['context_flip_rate']*100:.1f}%")
print(f"Average Latency:   {overall['average_latency_ms']} ms")

### Step 7 — Generate Decision Gate Report

In [ ]:
from src.evaluation.decision_gate import generate_decision_gate_report

report_path = 'reports/colab_decision_gate_report.md'
report_content = generate_decision_gate_report(overall, pair_m, failures, report_path)
print(f"Decision Gate Report generated at: {report_path}")